In [1]:
import os
import sys
sys.path.append('..')

from src.rag_pipeline import build_vectorstore, rag_pipeline

c:\Users\liauw\Desktop\Sputnik\2025-26\Courses\block-6\575-nlp\DSCI_575_project_cliauwyt_cea\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


# Choose a model

In [2]:
from transformers import pipeline
generator = pipeline(
    task="text-generation",
    model="Qwen/Qwen3.5-0.8B",
    max_new_tokens=512
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# RAG Semantic

## Load vector store

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

corpus_path = '../data/processed/preprocessed_corpus.csv'
vector_path = "../data/processed/vector_store"

if not os.path.exists(vector_path):
    build_vectorstore(corpus_path, vector_path, embeddings)
    print(f"Saved vector store to {vector_path}")
    
vectorstore = FAISS.load_local(
    vector_path, embeddings, allow_dangerous_deserialization=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval

In [5]:
retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}  # Fetch k most similar documents
    )
query = "what is the best soap"
retriever.invoke(query)

[Document(id='39aa2e66-1341-4f44-8bec-a3f258152067', metadata={'source': 'data/processed/preprocessed_corpus.csv', 'row': 6054, 'asin': 'B0716PQVP2', 'product_title': 'Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool', 'rating': '5.0', 'review_text': 'Must have with bars of soap ! You will love !'}, page_content='text: double layer exfoliating mesh soap saver pouch bubble foam net handmade soap mesh bag body facial cleaning tool health personal care great bar soap love'),
 Document(id='b3cffd8e-54ae-413b-8758-c607965b7230', metadata={'source': 'data/processed/preprocessed_corpus.csv', 'row': 5693, 'asin': 'B08DV37PZV', 'product_title': 'Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack', 'rating': '5.0', 'review_text': 'That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.'}, page_content='text: palmolive ultra original dish liquid pack 

## Pipeline

In [5]:
prompt = """You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

    Customer Reviews: {context}

    Question: {query}

    Answer based on the reviews above:"""


rag = rag_pipeline(vectorstore, generator, query, prompt)
print(rag)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.

    Customer Reviews: Product ASIN: B0716PQVP2
Title: Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool
Rating: 5.0/5.0
Review: Must have with bars of soap ! You will love !


Product ASIN: B08DV37PZV
Title: Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack
Rating: 5.0/5.0
Review: That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.


Product ASIN: B001HDZT7I
Title: Travelon Hand Soap Toiletry Sheets, 50-Count
Rating: 3.0/5.0
Review: Although this had decent reviews when I researched I learned very quickly that once these are wet they are not usable. The sheet size maybe can cover hand washing but they clump up and there isn't a lather like with ba

In [6]:
prompt = """You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    Be concise and direct. Never repeat yourself.
    
    Customer Reviews: {context}

    Question: {query}

    Answer based on the reviews above:"""


rag = rag_pipeline(vectorstore, generator, query, prompt)
print(rag)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    Be concise and direct. Never repeat yourself.

    Customer Reviews: Product ASIN: B0716PQVP2
Title: Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool
Rating: 5.0/5.0
Review: Must have with bars of soap ! You will love !


Product ASIN: B08DV37PZV
Title: Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack
Rating: 5.0/5.0
Review: That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.


Product ASIN: B001HDZT7I
Title: Travelon Hand Soap Toiletry Sheets, 50-Count
Rating: 3.0/5.0
Review: Although this had decent reviews when I researched I learned very quickly that once these are wet they are not usable. The sheet size maybe can cover hand washing but t

In [8]:
prompt = """You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    Be concise and direct. Never repeat yourself.
    If reviews don't contain enough info, say so — don't speculate.
    
    Customer Reviews: {context}

    Question: {query}

    Answer based on the reviews above:"""


rag = rag_pipeline(vectorstore, generator, query, prompt)
print(rag)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    Be concise and direct. Never repeat yourself.
    If reviews don't contain enough info, say so — don't speculate.

    Customer Reviews: Product ASIN: B0716PQVP2
Title: Dealglad 10Pcs Double Layer Exfoliating Mesh Soap Saver Pouch Bubble Foam Net Handmade Soap Mesh Bag Body Facial Cleaning Tool
Rating: 5.0/5.0
Review: Must have with bars of soap ! You will love !


Product ASIN: B08DV37PZV
Title: Palmolive Ultra Original Dish Liquid, 102 fl. oz. - 2 Pack
Rating: 5.0/5.0
Review: That "blue" dish soap is more difficult to rinse off.  I like Palmolive because it cleans well and rinses off easily.


Product ASIN: B001HDZT7I
Title: Travelon Hand Soap Toiletry Sheets, 50-Count
Rating: 3.0/5.0
Review: Although this had decent reviews when I researched I learned very quickly that once these are wet th